In [31]:
!pip install -q torch torchvision onnx onnxruntime onnxruntime-tools onnxscript

In [32]:
import torch
import torch.nn as nn
import onnx

from torchvision.models import mobilenet_v3_small
from onnxruntime.quantization import quantize_dynamic, QuantType

In [33]:
# Create MobileNetV3 Small architecture
model = mobilenet_v3_small(weights=None)

# Replace the classifier for 9 classes
model.classifier[3] = nn.Linear(
    model.classifier[3].in_features,
    9
)

# Load trained weights
model.load_state_dict(
    torch.load(
        "best_mobilenetv3_small.pth",
        map_location="cpu"
    )
)

model.eval()

print("✓ Model loaded successfully")

✓ Model loaded successfully


In [34]:
dummy_input = torch.randn(1, 3, 224, 224)

In [36]:
torch.onnx.export(
    model,
    dummy_input,
    "mobilenetv3_small_fp32.onnx",

    export_params=True,
    opset_version=13,
    do_constant_folding=True,

    input_names=["input"],
    output_names=["output"],

    dynamo=False          # Legacy exporter
)

print("✓ FP32 ONNX exported")

/tmp/ipykernel_1090/984203522.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✓ FP32 ONNX exported


In [37]:
onnx_model = onnx.load("mobilenetv3_small_fp32.onnx")

onnx.checker.check_model(onnx_model)

print("✓ ONNX model is valid")

✓ ONNX model is valid


In [38]:
quantize_dynamic(
    model_input="mobilenetv3_small_fp32.onnx",
    model_output="mobilenetv3_small_int8.onnx",
    weight_type=QuantType.QInt8
)

print("✓ INT8 ONNX created")

✓ INT8 ONNX created
